In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)

# Загрузка
df = pd.read_csv('../loan_data.csv')
print(f"Размер данных: {df.shape}")
df.head()

In [ ]:
print("=== Информация о данных ===\n")
df.info()

print("\n=== Статистика числовых признаков ===\n")
df.describe()

In [ ]:
missing = df.isnull().sum()
missing_percent = 100 * missing / len(df)
missing_df = pd.DataFrame({'Пропуски': missing, 'Процент': missing_percent})
missing_df[missing_df['Пропуски'] > 0]

In [ ]:
print("=== Распределение loan_status ===\n")
print(df['loan_status'].value_counts())
print(f"\nДоля одобрений: {df['loan_status'].mean():.2%}")
print(f"Доля отказов: {1 - df['loan_status'].mean():.2%}")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['loan_status'].value_counts().plot(kind='bar', ax=axes[0], title='Count')
df['loan_status'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], title='Percentage', autopct='%1.1f%%')
plt.tight_layout()
plt.show()

In [ ]:
# Числовые колонки для анализа
numeric_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 
                'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 
                'credit_score']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df[col].hist(ax=axes[i], bins=50, edgecolor='black')
    axes[i].set_title(col)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='mean')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label='median')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Явные выбросы
print("=== Подозрительные значения ===\n")
print(f"Максимальный возраст: {df['person_age'].max()} лет")
print(f"Максимальный стаж: {df['person_emp_exp'].max()} лет")
print(f"Минимальная ставка: {df['loan_int_rate'].min()}%")
print(f"Минимальный кредитный рейтинг: {df['credit_score'].min()}")

In [ ]:
# Кодируем категориальные для корреляции
df_encoded = pd.get_dummies(df, columns=['person_gender', 'person_education', 
                                          'person_home_ownership', 'loan_intent'])
df_encoded['previous_defaults'] = df_encoded['previous_loan_defaults_on_file'].map({'Yes': 1, 'No': 0})

# Корреляция с target
correlations = df_encoded.corr()['loan_status'].sort_values(ascending=False)
print("=== Топ-15 признаков по корреляции с loan_status ===\n")
print(correlations.head(15))

In [ ]:
categorical_cols = ['person_gender', 'person_education', 'person_home_ownership', 
                    'loan_intent', 'previous_loan_defaults_on_file']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if i < len(categorical_cols):
        df.groupby(col)['loan_status'].mean().sort_values().plot(kind='barh', ax=axes[i])
        axes[i].set_title(f'{col} - approval rate')
        axes[i].set_xlabel('Approval Rate')

plt.tight_layout()
plt.show()

In [ ]:
print("=== ВЫВОДЫ ПО EDA ===\n")
print("1. Размер данных: X строк, X признаков")
print("2. Пропуски: есть/нет")
print("3. Дисбаланс классов: X% одобрений, X% отказов")
print("4. Выбросы: нужно удалить age > 100 и emp_exp > age")
print("5. Сильные признаки: credit_score, previous_defaults, loan_percent_income")
print("6. Категориальные признаки: все имеют смысл, оставляем")